In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, chi2
from sksurv.metrics import concordance_index_censored
import math
import seaborn as sns

pd.options.mode.chained_assignment = None

# 67_33

## 建立資料集

In [ ]:
df = pd.read_csv("67_33_nandel_240226.csv")
#mask = (df.Derivation_or_Validation_group != 0)
# df = df.drop("Unnamed: 0",axis=1)
df.drop("Unnamed: 0",axis=1,inplace=True)
# df.drop("ETV1TDFTAF2",axis=1,inplace=True)
df["HCCdurationwks"] = df["HCCduration (wks)"]
df.drop("HCCduration (wks)",axis=1,inplace=True)
print(df.columns)


## PAGE-B

In [ ]:
feature = ["Derivation_or_Validation_group","age1yr","sex1M0F","plateletR","HCC", "HCCdurationwks"]
PAGE_B_test = df[feature]
PAGE_B_test["age"] = PAGE_B_test["age1yr"]-1
PAGE_B_test.drop("age1yr",axis=1,inplace=True)
PAGE_B_test["plateletR"] = PAGE_B_test["plateletR"]*1000
PAGE_B_test

In [ ]:
for i in range(PAGE_B_test["Derivation_or_Validation_group"].count()):
    if(PAGE_B_test.loc[i,"Derivation_or_Validation_group"] != 0.0):
        score = 0
        if (PAGE_B_test.loc[i,"age"]<=29 and PAGE_B_test.loc[i,"age"]>=16):
            pass
        elif(PAGE_B_test.loc[i,"age"]<=39 and PAGE_B_test.loc[i,"age"]>=30):
            score+=2
        elif(PAGE_B_test.loc[i,"age"]<=49 and PAGE_B_test.loc[i,"age"]>=40):
            score+=4
        elif(PAGE_B_test.loc[i,"age"]<=59 and PAGE_B_test.loc[i,"age"]>=50):
            score+=6
        elif(PAGE_B_test.loc[i,"age"]<=69 and PAGE_B_test.loc[i,"age"]>=60):
            score+=8
        elif(PAGE_B_test.loc[i,"age"]>=70):
            score+=10
        '''-------------------------------------------'''
        if(PAGE_B_test.loc[i,"sex1M0F"]==1):
            score+=6
        else:
            pass
        '''-------------------------------------------'''
        if(PAGE_B_test.loc[i,"plateletR"]>=200000):
            pass
        elif(PAGE_B_test.loc[i,"plateletR"]<=199999 and PAGE_B_test.loc[i,"plateletR"]>=100000):
            score+=6
        else:
            score+=9
        PAGE_B_test.loc[i,"PAGE_B_score"] = score
PAGE_B_test

In [ ]:
df["PAGE-B"] =  PAGE_B_test["PAGE_B_score"]
df

In [ ]:
df.to_csv("./compare/67_33ModelP_score.csv")

In [ ]:
#trainset c-index test
trainmask = (PAGE_B_test["Derivation_or_Validation_group"]==1)
PAGE_B_train_test = PAGE_B_test[trainmask]
PAGE_B_train_test.loc[:,'HCC'] = PAGE_B_train_test['HCC'].map({1:True,0:False})
cindex = concordance_index_censored(
    PAGE_B_train_test['HCC'],
    PAGE_B_train_test['HCCdurationwks'],
    PAGE_B_train_test["PAGE_B_score"], 
)
print(round(cindex[0], 3))

In [ ]:
#testset c-index test
testmask = (PAGE_B_test["Derivation_or_Validation_group"]==2)
PAGE_B_test_test = PAGE_B_test[testmask]
PAGE_B_test_test.loc[:,'HCC'] = PAGE_B_test_test['HCC'].map({1:True,0:False})
cindex = concordance_index_censored(
    PAGE_B_test_test['HCC'],
    PAGE_B_test_test['HCCdurationwks'],
    PAGE_B_test_test["PAGE_B_score"], 
)
print(round(cindex[0], 3))

In [ ]:
allmask = (PAGE_B_test["Derivation_or_Validation_group"]!=0)
PAGE_B_all_test = PAGE_B_test[allmask]
PAGE_B_all_test.loc[:,'HCC'] = PAGE_B_all_test['HCC'].map({1:True,0:False})
cindex = concordance_index_censored(
    PAGE_B_all_test['HCC'],
    PAGE_B_all_test['HCCdurationwks'],
    PAGE_B_all_test["PAGE_B_score"], 
)
print(round(cindex[0], 3))

## mPAGE-B

In [ ]:
feature = ["Derivation_or_Validation_group","age1yr","sex1M0F","plateletR","albuminR","HCC", "HCCdurationwks"]
mPAGE_B_test = df[feature]
mPAGE_B_test["age"] = mPAGE_B_test["age1yr"]-1
mPAGE_B_test.drop("age1yr",axis=1,inplace=True)
mPAGE_B_test

In [ ]:
for i in range(mPAGE_B_test["Derivation_or_Validation_group"].count()):
    if(mPAGE_B_test.loc[i,"Derivation_or_Validation_group"] != 0.0):
        score = 0
        if (mPAGE_B_test.loc[i,"age"]<30):
            pass
        elif(mPAGE_B_test.loc[i,"age"]<40 and mPAGE_B_test.loc[i,"age"]>=30):
            score+=3
        elif(mPAGE_B_test.loc[i,"age"]<50 and mPAGE_B_test.loc[i,"age"]>=40):
            score+=5
        elif(mPAGE_B_test.loc[i,"age"]<60 and mPAGE_B_test.loc[i,"age"]>=50):
            score+=7
        elif(mPAGE_B_test.loc[i,"age"]<70 and mPAGE_B_test.loc[i,"age"]>=60):
            score+=9
        elif(mPAGE_B_test.loc[i,"age"]>=70):
            score+=11
        '''----------------------------------------------------'''
        if(mPAGE_B_test.loc[i,"sex1M0F"]==1):
            score+=2
        else:
            pass
        '''----------------------------------------------------'''
        if(mPAGE_B_test.loc[i,"plateletR"]>=250):
            pass
        elif(mPAGE_B_test.loc[i,"plateletR"]<250 and mPAGE_B_test.loc[i,"plateletR"]>=200):
            score+=2
        elif(mPAGE_B_test.loc[i,"plateletR"]<200 and mPAGE_B_test.loc[i,"plateletR"]>=150):
            score+=3
        elif(mPAGE_B_test.loc[i,"plateletR"]<150 and mPAGE_B_test.loc[i,"plateletR"]>=100):
            score+=4
        else:
            score+=5
        '''----------------------------------------------------'''
        if(mPAGE_B_test.loc[i,"albuminR"]>=4):
            pass
        elif(mPAGE_B_test.loc[i,"albuminR"]<4 and mPAGE_B_test.loc[i,"albuminR"]>=3.5):
            score+=1
        elif(mPAGE_B_test.loc[i,"albuminR"]<3.5 and mPAGE_B_test.loc[i,"albuminR"]>=3):
            score+=2
        else:
            score+=3
        mPAGE_B_test.loc[i,"mPAGE_B_score"] = score
mPAGE_B_test

In [ ]:
df["mPAGE-B"] =  mPAGE_B_test["mPAGE_B_score"]
df

In [ ]:
df.to_csv("./compare/67_33ModelmP_score.csv")

In [ ]:
#trainset c-index test
trainmask = (mPAGE_B_test["Derivation_or_Validation_group"]==1)
mPAGE_B_train_test = mPAGE_B_test[trainmask]
mPAGE_B_train_test.loc[:,'HCC'] = mPAGE_B_train_test['HCC'].map({1:True,0:False})
cindex = concordance_index_censored(
    mPAGE_B_train_test['HCC'],
    mPAGE_B_train_test['HCCdurationwks'],
    mPAGE_B_train_test["mPAGE_B_score"], 
)
print(round(cindex[0], 3))

In [ ]:
#testset c-index test
testmask = (mPAGE_B_test["Derivation_or_Validation_group"]==2)
mPAGE_B_test_test = mPAGE_B_test[testmask]
mPAGE_B_test_test.loc[:,'HCC'] = mPAGE_B_test_test['HCC'].map({1:True,0:False})
cindex = concordance_index_censored(
    mPAGE_B_test_test['HCC'],
    mPAGE_B_test_test['HCCdurationwks'],
    mPAGE_B_test_test["mPAGE_B_score"], 
)
print(round(cindex[0], 3))

In [ ]:
allmask = (mPAGE_B_test["Derivation_or_Validation_group"]!=0)
mPAGE_B_all_test = mPAGE_B_test[allmask]
mPAGE_B_all_test.loc[:,'HCC'] = mPAGE_B_all_test['HCC'].map({1:True,0:False})
cindex = concordance_index_censored(
    mPAGE_B_all_test['HCC'],
    mPAGE_B_all_test['HCCdurationwks'],
    mPAGE_B_all_test["mPAGE_B_score"], 
)
print(round(cindex[0], 3))

## REACH-B

In [ ]:
feature = ["Derivation_or_Validation_group","sex1M0F","age1yr","gpt","HBeAg","logHBVDNA","HCC", "HCCdurationwks"]
REACH_B_test = df[feature]
REACH_B_test["age"] = REACH_B_test["age1yr"]-1
REACH_B_test.drop("age1yr",axis=1,inplace=True)
REACH_B_test

In [ ]:
for i in range(REACH_B_test["Derivation_or_Validation_group"].count()):
    REACH_B_test.loc[i,"HBVDNA"] = math.pow(10,(REACH_B_test.loc[i,"logHBVDNA"]))*5.6
REACH_B_test

In [ ]:
for i in range(REACH_B_test["Derivation_or_Validation_group"].count()):
    if(REACH_B_test.loc[i,"Derivation_or_Validation_group"] != 0.0):
        score = 0
        if (REACH_B_test.loc[i,"age"]<35): #評分表上面從30歲開始，但30-34歲還是0分，所以我把35歲以下全評成0分
            pass
        elif(REACH_B_test.loc[i,"age"]<40 and REACH_B_test.loc[i,"age"]>=35):
            score+=1
        elif(REACH_B_test.loc[i,"age"]<45 and REACH_B_test.loc[i,"age"]>=40):
            score+=2
        elif(REACH_B_test.loc[i,"age"]<50 and REACH_B_test.loc[i,"age"]>=45):
            score+=3
        elif(REACH_B_test.loc[i,"age"]<55 and REACH_B_test.loc[i,"age"]>=50):
            score+=4
        elif(REACH_B_test.loc[i,"age"]<60 and REACH_B_test.loc[i,"age"]>=55):
            score+=5
        elif(REACH_B_test.loc[i,"age"]>=60): #評分表最多到65歲，60-65歲是6分，所以我把60歲以上全評成6分
            score+=6
        '''-------------------------------------------'''
        if(REACH_B_test.loc[i,"sex1M0F"]==1):
            score+=2
        else:
            pass
        '''-------------------------------------------'''
        if(REACH_B_test.loc[i,"gpt"]<15):
            pass
        elif(REACH_B_test.loc[i,"gpt"]>=15 and REACH_B_test.loc[i,"gpt"]<45):
            score+=1
        else:
            score+=2
        '''-------------------------------------------'''
        if(REACH_B_test.loc[i,"HBeAg"]==1):
            score+=2
        else:
            pass
        '''-------------------------------------------'''
        if(REACH_B_test.loc[i,"HBVDNA"]<10000):
            pass
        elif(REACH_B_test.loc[i,"HBVDNA"]>=10000 and REACH_B_test.loc[i,"HBVDNA"]<100000):
            score+=3
        elif(REACH_B_test.loc[i,"HBVDNA"]>=100000 and REACH_B_test.loc[i,"HBVDNA"]<1000000):
            score+=5
        else:
            score+=4
        REACH_B_test.loc[i,"REACH_B_score"] = score
mPAGE_B_test

In [ ]:
df["REACH-B"] =  REACH_B_test["REACH_B_score"]
df

In [ ]:
df.to_csv("./compare/67_33ModelmPR_score.csv")

In [ ]:
#trainset c-index test
trainmask = (REACH_B_test["Derivation_or_Validation_group"]==1)
REACH_B_train_test = REACH_B_test[trainmask]
REACH_B_train_test.loc[:,'HCC'] = REACH_B_train_test['HCC'].map({1:True,0:False})
cindex = concordance_index_censored(
    REACH_B_train_test['HCC'],
    REACH_B_train_test['HCCdurationwks'],
    REACH_B_train_test["REACH_B_score"], 
)
print(round(cindex[0], 3))

In [ ]:
#testset c-index test
testmask = (REACH_B_test["Derivation_or_Validation_group"]==2)
REACH_B_test_test = REACH_B_test[testmask]
REACH_B_test_test.loc[:,'HCC'] = REACH_B_test_test['HCC'].map({1:True,0:False})
cindex = concordance_index_censored(
    REACH_B_test_test['HCC'],
    REACH_B_test_test['HCCdurationwks'],
    REACH_B_test_test["REACH_B_score"], 
)
print(round(cindex[0], 3))

In [ ]:
allmask = (REACH_B_test["Derivation_or_Validation_group"]!=0)
REACH_B_all_test = REACH_B_test[allmask]
REACH_B_all_test.loc[:,'HCC'] = REACH_B_all_test['HCC'].map({1:True,0:False})
cindex = concordance_index_censored(
    REACH_B_all_test['HCC'],
    REACH_B_all_test['HCCdurationwks'],
    REACH_B_all_test["REACH_B_score"], 
)
print(round(cindex[0], 3))

## PLAN-B

In [ ]:
PLAN_B = df.copy()
PLAN_B_test = PLAN_B[["Derivation_or_Validation_group","age1yr"]]
PLAN_B_test["age"] = PLAN_B_test["age1yr"]-1
PLAN_B_test.drop("age1yr",axis=1,inplace=True)
PLAN_B_test["male"] = PLAN_B["sex1M0F"]
PLAN_B_test["e_t"] = PLAN_B["ETV1TDFTAF2"]
PLAN_B_test["lc"] = 1
PLAN_B_test["plt"] = PLAN_B["plateletR"]
PLAN_B_test["alb"] = PLAN_B["albuminR"]
PLAN_B_test["tbil"] = PLAN_B["t-bil"]
PLAN_B_test["alt"] = PLAN_B["gpt"]
PLAN_B_test["dna"] = PLAN_B["logHBVDNA"]
PLAN_B_test["hbeag"] = PLAN_B["HBeAg"]
PLAN_B_test["HCC"] = PLAN_B["HCC"]
PLAN_B_test["HCCdurationwks"] = PLAN_B["HCCdurationwks"]
PLAN_B_test["codenumber"] = PLAN_B["codenumber"]
PLAN_B_test

In [ ]:
mask1 = (PLAN_B_test.Derivation_or_Validation_group==1)
mask2 = (PLAN_B_test.Derivation_or_Validation_group==2)
trainman = PLAN_B_test[mask1].reset_index(drop=True)
print("推導隊列有",trainman["age"].count(),"人")
testman = PLAN_B_test[mask2].reset_index(drop=True)
print("驗證隊列有",testman["age"].count(),"人")
print("推導隊列佔",trainman["age"].count()/(trainman["age"].count()+testman["age"].count()))

trainHCClist = (trainman.HCC==1)
trainHCC = trainman[trainHCClist]
print("訓練資料集數量為",len(trainman),"，其中罹患肝癌患者有",len(trainHCC),"人，佔訓練集比例為",len(trainHCC)/len(trainman))
testHCClist = (testman.HCC==1)
testHCC = testman[testHCClist]
print("測試資料集數量為",len(testman),"，其中罹患肝癌患者有",len(testHCC),"人，佔測試集比例為",len(testHCC)/len(testman))

In [ ]:
PLAN_B_test_order = PLAN_B_test[["Derivation_or_Validation_group","male","age","e_t","lc","plt","alb","tbil","alt","dna","hbeag","HCC","HCCdurationwks","codenumber"]]
PLAN_B_test_order

In [ ]:
!wget https://github.com/vitaldb/hbvhcc/raw/main/model.json

In [ ]:
model = xgb.Booster()
model.load_model('model.json')
inxgb = (PLAN_B_test_order["Derivation_or_Validation_group"]!=0)
PLAN_B_test = PLAN_B_test_order[inxgb] #後面不會用到，取代掉沒關係
X = PLAN_B_test.drop(["Derivation_or_Validation_group","HCC","HCCdurationwks","codenumber"],axis=1).astype(float).values
dtest_reg = xgb.DMatrix(X, PLAN_B_test['HCC'])
PLAN_B_test['PLAN_B_score'] = model.predict(dtest_reg)
PLAN_B_test

In [ ]:
merge = PLAN_B_test[["codenumber","PLAN_B_score"]]
df = pd.merge(df,merge,how='outer')
df

In [ ]:
df.to_csv("./compare/67_33ModelmPRL_score.csv")

In [ ]:
#trainset c-index test
trainmask = (PLAN_B_test["Derivation_or_Validation_group"]==1)
PLAN_B_train = PLAN_B_test[trainmask]
PLAN_B_train.loc[:,'HCC'] = PLAN_B_train['HCC'].map({1:True,0:False})
cindex = concordance_index_censored(
    PLAN_B_train['HCC'],
    PLAN_B_train['HCCdurationwks'],
    PLAN_B_train["PLAN_B_score"], 
)
print(round(cindex[0], 3))

In [ ]:
#testset c-index test
testmask = (PLAN_B_test["Derivation_or_Validation_group"]==2)
PLAN_B_test_test = PLAN_B_test[testmask]
PLAN_B_test_test.loc[:,'HCC'] = PLAN_B_test_test['HCC'].map({1:True,0:False})
cindex = concordance_index_censored(
    PLAN_B_test_test['HCC'],
    PLAN_B_test_test['HCCdurationwks'],
    PLAN_B_test_test["PLAN_B_score"], 
)
print(round(cindex[0], 3))

In [ ]:
allmask = (PLAN_B_test["Derivation_or_Validation_group"]!=0)
PLAN_B_all = PLAN_B_test[allmask]
PLAN_B_all.loc[:,'HCC'] = PLAN_B_all['HCC'].map({1:True,0:False})
cindex = concordance_index_censored(
    PLAN_B_all['HCC'],
    PLAN_B_all['HCCdurationwks'],
    PLAN_B_all["PLAN_B_score"], 
)
print(round(cindex[0], 3))

## PLAN-B-Retrain

In [ ]:
PLAN_B = df.copy()
PLAN_B_test = PLAN_B[["Derivation_or_Validation_group","age1yr"]]
PLAN_B_test["age"] = PLAN_B_test["age1yr"]-1
PLAN_B_test.drop("age1yr",axis=1,inplace=True)
PLAN_B_test["male"] = PLAN_B["sex1M0F"]
PLAN_B_test["e_t"] = PLAN_B["ETV1TDFTAF2"]
PLAN_B_test["lc"] = 1
PLAN_B_test["plt"] = PLAN_B["plateletR"]
PLAN_B_test["alb"] = PLAN_B["albuminR"]
PLAN_B_test["tbil"] = PLAN_B["t-bil"]
PLAN_B_test["alt"] = PLAN_B["gpt"]
PLAN_B_test["dna"] = PLAN_B["logHBVDNA"]
PLAN_B_test["hbeag"] = PLAN_B["HBeAg"]
PLAN_B_test["HCC"] = PLAN_B["HCC"]
PLAN_B_test["HCCdurationwks"] = PLAN_B["HCCdurationwks"]
PLAN_B_test["codenumber"] = PLAN_B["codenumber"]
PLAN_B_test

In [ ]:
PLAN_B_test_order = PLAN_B_test[["Derivation_or_Validation_group","male","age","e_t","lc","plt","alb","tbil","alt","dna","hbeag","HCC","HCCdurationwks","codenumber"]]
PLAN_B_test_order

In [ ]:
trainmask = (PLAN_B_test_order["Derivation_or_Validation_group"]==1)
train = PLAN_B_test_order[trainmask]
print(len(train.index))

In [ ]:
X_train = train.drop(["HCC","HCCdurationwks","codenumber","Derivation_or_Validation_group"],axis=1)
y_train = train[["HCC","HCCdurationwks","codenumber"]]
print(X_train)

In [ ]:
print(X_train.describe)

In [ ]:
dtrain_reg = xgb.DMatrix(X_train, y_train['HCC'])
params = {
        'booster': 'gbtree',
        'objective': 'binary:logistic',
        'eta': 0.07,
        'max_depth': 3,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'tree_method': 'exact',
        'seed': 1000,
        'scale_pos_weight':5,
        }

In [ ]:
num_rounds = 50
plst = list(params.items())
model = xgb.train(plst, dtrain_reg, num_rounds)

In [ ]:
testmask = (PLAN_B_test_order["Derivation_or_Validation_group"]==2)
test = PLAN_B_test_order[testmask]
X_test = test.drop(["HCC","HCCdurationwks","codenumber","Derivation_or_Validation_group"],axis=1)
y_test = test[["HCC","HCCdurationwks","codenumber"]]
dtest_reg = xgb.DMatrix(X_test, y_test['HCC'])

In [ ]:
y_test.loc[:,'pred'] = model.predict(dtest_reg)
print(y_test)

In [ ]:
y_test.loc[:,'HCC'] = y_test['HCC'].map({1:True,0:False})
cindex = concordance_index_censored(
    y_test['HCC'],
    y_test['HCCdurationwks'],
    y_test["pred"],
)
print(round(cindex[0], 3))
y_test['HCC'] = y_test['HCC'].map({True:1,False:0})

In [ ]:
y_train.loc[:,'pred'] = model.predict(dtrain_reg)
print(y_train)

In [ ]:
y_train.loc[:,'HCC'] = y_train['HCC'].map({1:True,0:False})
cindex = concordance_index_censored(
    y_train['HCC'],
    y_train['HCCdurationwks'],
    y_train["pred"], 
)
print(round(cindex[0], 3))
y_train['HCC'] = y_train['HCC'].map({True:1,False:0})

In [ ]:
model.save_model('./compare/67_33model_PLANB_retrain.json')

In [ ]:
tryy = df
ttt = pd.merge(tryy,y_train[["codenumber","pred"]],on="codenumber",how="outer")
ttt

In [ ]:
xxx = pd.merge(ttt,y_test[["codenumber","pred"]],on="codenumber",how="outer")
xxx

In [ ]:
predmerge = xxx.copy()
for i in range(predmerge["codenumber"].count()):
    if (predmerge.loc[i,"pred_x"]>=0):
        predmerge.loc[i,"PLAN_B_retrain_score"] = predmerge.loc[i,"pred_x"]
    else:
        predmerge.loc[i,"PLAN_B_retrain_score"] = predmerge.loc[i,"pred_y"]
df["PLAN_B_retrain_score"] = predmerge["PLAN_B_retrain_score"]
df

In [ ]:
df.to_csv("./compare/67_33ModelmPRLr.csv")

## APA-B

In [ ]:
df = pd.read_csv("./compare/67_33ModelmPRLr.csv")
#mask = (df.Derivation_or_Validation_group != 0)
df = df.drop("Unnamed: 0",axis=1)
df.columns

In [ ]:
feature = ["Derivation_or_Validation_group","age1yr","platelet1yrR","afp1yrR","HCC", "HCCdurationwks"]
APA_B = df[feature]
APA_B

In [ ]:
for i in range(APA_B["Derivation_or_Validation_group"].count()):
    if(APA_B.loc[i,"Derivation_or_Validation_group"] != 0):
        score = 0
        if (APA_B.loc[i,"age1yr"]<40):
            pass
        elif(APA_B.loc[i,"age1yr"]<50 and APA_B.loc[i,"age1yr"]>=40):
            score+=1
        elif(APA_B.loc[i,"age1yr"]<60 and APA_B.loc[i,"age1yr"]>=50):
            score+=2
        elif(APA_B.loc[i,"age1yr"]<70 and APA_B.loc[i,"age1yr"]>=60):
            score+=3
        elif(APA_B.loc[i,"age1yr"]>=70):
            score+=4
        '''-----------------------------------------------------'''
        if(APA_B.loc[i,"platelet1yrR"]>=130):
            pass
        elif(APA_B.loc[i,"platelet1yrR"]<130 and APA_B.loc[i,"platelet1yrR"]>=100):
            score+=3
        else:
            score+=6
        '''-----------------------------------------------------'''
        if(APA_B.loc[i,"afp1yrR"] < 5):
            pass
        elif(APA_B.loc[i,"afp1yrR"]>=5 and APA_B.loc[i,"afp1yrR"]<=9):
            score+=2
        else:
            score+=5
        APA_B.loc[i,"APA_B_score"] = score
APA_B

In [ ]:
df["APA-B"] =  APA_B["APA_B_score"]
df

In [ ]:
df.to_csv("./compare/67_33ModelmPRLrA_score.csv")

In [ ]:
#trainset c-index test
trainmask = (APA_B["Derivation_or_Validation_group"]==1)
APA_B_train = APA_B[trainmask]
APA_B_train.loc[:,'HCC'] = APA_B_train['HCC'].map({1:True,0:False})
cindex = concordance_index_censored(
    APA_B_train['HCC'],
    APA_B_train['HCCdurationwks'],
    APA_B_train["APA_B_score"], 
)
print(round(cindex[0], 3))

In [ ]:
#testset c-index test
testmask = (APA_B["Derivation_or_Validation_group"]==2)
APA_B_test = APA_B[testmask]
APA_B_test.loc[:,'HCC'] = APA_B_test['HCC'].map({1:True,0:False})
cindex = concordance_index_censored(
    APA_B_test['HCC'],
    APA_B_test['HCCdurationwks'],
    APA_B_test["APA_B_score"], 
)
print(round(cindex[0], 3))

In [ ]:
allmask = (APA_B["Derivation_or_Validation_group"]!=0)
APA_B_all = APA_B[allmask]
APA_B_all.loc[:,'HCC'] = APA_B_all['HCC'].map({1:True,0:False})
cindex = concordance_index_censored(
    APA_B_all['HCC'],
    APA_B_all['HCCdurationwks'],
    APA_B_all["APA_B_score"], 
)
print(round(cindex[0], 3))

## CU-HCC

In [ ]:
df = pd.read_csv("./compare/67_33ModelmPRLrA_score.csv")
#mask = (df.Derivation_or_Validation_group != 0)
df = df.drop("Unnamed: 0",axis=1)
df.columns

In [ ]:
feature = ["Derivation_or_Validation_group","age1yr","albuminR","t-bil","logHBVDNA","HCC", "HCCdurationwks"]
CU_HCC = df[feature]
CU_HCC["age"] = CU_HCC["age1yr"]-1
CU_HCC.drop("age1yr",axis=1,inplace=True)
CU_HCC["Cirrhosis"] = 1
CU_HCC["albuminR"] = CU_HCC["albuminR"]*10
CU_HCC["t-bil"] = CU_HCC["t-bil"]*17.1
CU_HCC

In [ ]:
for i in range(CU_HCC["Derivation_or_Validation_group"].count()):
    mid = math.pow(10,(CU_HCC.loc[i,"logHBVDNA"]))
    CU_HCC.loc[i,"logDNAumol"] = math.log10(mid*5.6)
CU_HCC

In [ ]:
for i in range(CU_HCC["Derivation_or_Validation_group"].count()):
    if(CU_HCC.loc[i,"Derivation_or_Validation_group"] != 0):
        score = 0
        if (CU_HCC.loc[i,"age"]>50):
            score+=3
        else:
            pass
        '''------------------------------------------------'''
        if(CU_HCC.loc[i,"albuminR"]<=35):
            score+=20
        else:
            pass
        '''------------------------------------------------'''
        if(CU_HCC.loc[i,"t-bil"]>18):
            score+=1.5
        else:
            pass
        '''------------------------------------------------'''
        if(CU_HCC.loc[i,"logDNAumol"]<=4):
            pass
        elif(CU_HCC.loc[i,"logDNAumol"]<=6 and CU_HCC.loc[i,"logDNAumol"]>4):
            score+=1
        else:
            score+=4
        '''------------------------------------------------'''
        if(CU_HCC.loc[i,"Cirrhosis"]==1):
            score+=15
        else:
            pass
        CU_HCC.loc[i,"CU_HCC_score"] = score
CU_HCC

In [ ]:
df["CU-HCC"] =  CU_HCC["CU_HCC_score"]
df

In [ ]:
df.to_csv("./compare/67_33ModelmPRLrAU_score.csv")

In [ ]:
#trainset c-index test
trainmask = (CU_HCC["Derivation_or_Validation_group"]==1)
CU_HCC_train = CU_HCC[trainmask]
CU_HCC_train.loc[:,'HCC'] = CU_HCC_train['HCC'].map({1:True,0:False})
cindex = concordance_index_censored(
    CU_HCC_train['HCC'],
    CU_HCC_train['HCCdurationwks'],
    CU_HCC_train["CU_HCC_score"], 
)
print(round(cindex[0], 3))

In [ ]:
#testset c-index test
testmask = (CU_HCC["Derivation_or_Validation_group"]==2)
CU_HCC_test = CU_HCC[testmask]
CU_HCC_test.loc[:,'HCC'] = CU_HCC_test['HCC'].map({1:True,0:False})
cindex = concordance_index_censored(
    CU_HCC_test['HCC'],
    CU_HCC_test['HCCdurationwks'],
    CU_HCC_test["CU_HCC_score"], 
)
print(round(cindex[0], 3))

In [ ]:
allmask = (CU_HCC["Derivation_or_Validation_group"]!=0)
CU_HCC_all = CU_HCC[allmask]
CU_HCC_all.loc[:,'HCC'] = CU_HCC_all['HCC'].map({1:True,0:False})
cindex = concordance_index_censored(
    CU_HCC_all['HCC'],
    CU_HCC_all['HCCdurationwks'],
    CU_HCC_all["CU_HCC_score"], 
)
print(round(cindex[0], 3))

## Ours1yr

In [ ]:
import pandas as pd
from sksurv.metrics import concordance_index_censored
import seaborn as sns

df = pd.read_csv("./compare/67_33ModelmPRLrAU_score.csv")
#mask = (df.Derivation_or_Validation_group != 0)
df = df.drop("Unnamed: 0",axis=1)
df.columns

In [ ]:
import xgboost as xgb
trainmask = (df["Derivation_or_Validation_group"]==1)
feature = ['sex1M0F', 'ETV1TDFTAF2', 'HBVDNA1yr', 'age1yr', 'got1yr', 'T-bil1yrR', 'albuminR.1', 'platelet1yrR', 'afp1yrR']
X_train = df[trainmask][feature]
y_train = df[trainmask][["HCC","HCCdurationwks","codenumber"]]
dtrain_reg = xgb.DMatrix(X_train, y_train['HCC'])
testmask = (df["Derivation_or_Validation_group"]==2)
X_test = df[testmask][feature]
y_test = df[testmask][["HCC","HCCdurationwks","codenumber"]]
dtest_reg = xgb.DMatrix(X_test, y_test['HCC'])

model = xgb.Booster()
model.load_model("1YR754_xgboost_hyper.json")
print("feature:",X_train.columns)

y_train.loc[:,'pred'] = model.predict(dtrain_reg)
y_train.loc[:,'HCC'] = y_train['HCC'].map({1:True,0:False})
cindex = concordance_index_censored(
    y_train['HCC'],
    y_train['HCCdurationwks'],
    y_train["pred"], 
)
print("trainset C-index",round(cindex[0], 3))
y_train['HCC'] = y_train['HCC'].map({True:1,False:0})

y_test.loc[:,'pred'] = model.predict(dtest_reg)
y_test.loc[:,'HCC'] = y_test['HCC'].map({1:True,0:False})
cindex = concordance_index_censored(
    y_test['HCC'],
    y_test['HCCdurationwks'],
    y_test["pred"],
)
print("testset C-index",round(cindex[0], 3))
y_test['HCC'] = y_test['HCC'].map({True:1,False:0})

X_all = df[feature]
y_all = df[["HCC","HCCdurationwks","codenumber"]]
dall_reg = xgb.DMatrix(X_all,y_all["HCC"])

df.loc[:,'oursFirst'] = model.predict(dall_reg)
df.loc[:,'HCC'] = df['HCC'].map({1:True,0:False})
cindex = concordance_index_censored(
    df['HCC'],
    df['HCCdurationwks'],
    df["oursFirst"],
)
print("all C-index",round(cindex[0], 3))
df['HCC'] = df['HCC'].map({True:1,False:0})

In [ ]:
print(df)

In [ ]:
df.to_csv("./compare/67_33ModelmPRLrAUF.csv")

## OursALL

In [ ]:
import pandas as pd
from sksurv.metrics import concordance_index_censored
import seaborn as sns

df = pd.read_csv("./compare/67_33ModelmPRLrAUF.csv")
#mask = (df.Derivation_or_Validation_group != 0)
df = df.drop("Unnamed: 0",axis=1)
df.columns

In [ ]:
import xgboost as xgb
trainmask = (df["Derivation_or_Validation_group"]==1)
feature = ['albuminR', 'plateletR', 'age1yr', 'got1yr', 'platelet1yrR', 'afp1yrR']
X_train = df[trainmask][feature]
y_train = df[trainmask][["HCC","HCCdurationwks","codenumber"]]
dtrain_reg = xgb.DMatrix(X_train, y_train['HCC'])
testmask = (df["Derivation_or_Validation_group"]==2)
X_test = df[testmask][feature]
y_test = df[testmask][["HCC","HCCdurationwks","codenumber"]]
dtest_reg = xgb.DMatrix(X_test, y_test['HCC'])

model = xgb.Booster()
model.load_model("ALL761_xgboost_hyper.json")
print("feature:",X_train.columns)

y_train.loc[:,'pred'] = model.predict(dtrain_reg)
y_train.loc[:,'HCC'] = y_train['HCC'].map({1:True,0:False})
cindex = concordance_index_censored(
    y_train['HCC'],
    y_train['HCCdurationwks'],
    y_train["pred"], 
)
print("trainset C-index",round(cindex[0], 3))
y_train['HCC'] = y_train['HCC'].map({True:1,False:0})

y_test.loc[:,'pred'] = model.predict(dtest_reg)
y_test.loc[:,'HCC'] = y_test['HCC'].map({1:True,0:False})
cindex = concordance_index_censored(
    y_test['HCC'],
    y_test['HCCdurationwks'],
    y_test["pred"],
)
print("testset C-index",round(cindex[0], 3))
y_test['HCC'] = y_test['HCC'].map({True:1,False:0})

X_all = df[feature]
y_all = df[["HCC","HCCdurationwks","codenumber"]]
dall_reg = xgb.DMatrix(X_all,y_all["HCC"])

df.loc[:,'oursALL'] = model.predict(dall_reg)
df.loc[:,'HCC'] = df['HCC'].map({1:True,0:False})
cindex = concordance_index_censored(
    df['HCC'],
    df['HCCdurationwks'],
    df["oursALL"],
)
print("all C-index",round(cindex[0], 3))
df['HCC'] = df['HCC'].map({True:1,False:0})

In [ ]:
print(df)

In [ ]:
df.to_csv("./compare/67_33ModelmPRLrAUFA.csv")

## OursBASELINE

In [ ]:
import pandas as pd
from sksurv.metrics import concordance_index_censored
import seaborn as sns

df = pd.read_csv("./compare/67_33ModelmPRLrAUFA.csv")
#mask = (df.Derivation_or_Validation_group != 0)
df = df.drop("Unnamed: 0",axis=1)
df["age"]=df["age1yr"]-1
df.columns

In [ ]:
import xgboost as xgb
trainmask = (df["Derivation_or_Validation_group"]==1)
feature = ['sex1M0F', 'ETV1TDFTAF2', 'age', 'got', 'gpt', 't-bil', 'albuminR', 'plateletR']
X_train = df[trainmask][feature]
y_train = df[trainmask][["HCC","HCCdurationwks","codenumber"]]
dtrain_reg = xgb.DMatrix(X_train, y_train['HCC'])
testmask = (df["Derivation_or_Validation_group"]==2)
X_test = df[testmask][feature]
y_test = df[testmask][["HCC","HCCdurationwks","codenumber"]]
dtest_reg = xgb.DMatrix(X_test, y_test['HCC'])

model = xgb.Booster()
model.load_model("BASELINE709_xgboost_hyper.json")
print("feature:",X_train.columns)

y_train.loc[:,'pred'] = model.predict(dtrain_reg)
y_train.loc[:,'HCC'] = y_train['HCC'].map({1:True,0:False})
cindex = concordance_index_censored(
    y_train['HCC'],
    y_train['HCCdurationwks'],
    y_train["pred"], 
)
print("trainset C-index",round(cindex[0], 3))
y_train['HCC'] = y_train['HCC'].map({True:1,False:0})

y_test.loc[:,'pred'] = model.predict(dtest_reg)
y_test.loc[:,'HCC'] = y_test['HCC'].map({1:True,0:False})
cindex = concordance_index_censored(
    y_test['HCC'],
    y_test['HCCdurationwks'],
    y_test["pred"],
)
print("testset C-index",round(cindex[0], 3))
y_test['HCC'] = y_test['HCC'].map({True:1,False:0})

X_all = df[feature]
y_all = df[["HCC","HCCdurationwks","codenumber"]]
dall_reg = xgb.DMatrix(X_all,y_all["HCC"])

df.loc[:,'oursBASELINE'] = model.predict(dall_reg)
df.loc[:,'HCC'] = df['HCC'].map({1:True,0:False})
cindex = concordance_index_censored(
    df['HCC'],
    df['HCCdurationwks'],
    df["oursBASELINE"],
)
print("all C-index",round(cindex[0], 3))
df['HCC'] = df['HCC'].map({True:1,False:0})

In [ ]:
print(df)

In [ ]:
df.drop("age",axis=1,inplace=True)
df.to_csv("./compare/67_33ModelmPRLrAUFAB.csv")